# Assignment 2: IMDB Sentiment Classification
## Deep Neural Network for Binary Text Classification

In [ ]:
# Install required packages
!pip install tensorflow numpy pandas matplotlib seaborn scikit-learn chardet

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
# Load dataset
# IMPORTANT: Update the path to your CSV file
data = pd.read_csv('IMDB Dataset.csv', encoding='latin-1')

print("Dataset shape:", data.shape)
print("\nColumns:", data.columns.tolist())
print("\nSentiment distribution:")
print(data['sentiment'].value_counts())
print("\nFirst 3 reviews:")
print(data.head(3))

In [ ]:
# Prepare data
# Map sentiment to binary labels
data['label'] = data['sentiment'].map({'positive': 1, 'negative': 0})

reviews = data['review'].values
labels = data['label'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(reviews, labels, test_size=0.2, random_state=42, stratify=labels)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Positive samples in train: {np.sum(y_train)} ({np.mean(y_train)*100:.1f}%)")
print(f"Negative samples in train: {len(y_train) - np.sum(y_train)} ({(1-np.mean(y_train))*100:.1f}%)")

In [ ]:
# Tokenize text
max_words = 10000  # Vocabulary size
max_len = 256      # Maximum review length

print("Tokenizing text...")
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

# Convert text to sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

print(f"✓ Tokenization complete!")
print(f"Training data shape: {X_train_pad.shape}")
print(f"Test data shape: {X_test_pad.shape}")

In [ ]:
# Define hyperparameter configurations to test
configs = [
    {'name': 'Config 1: ReLU, Batch 32', 'activation': 'relu', 'batch_size': 32},
    {'name': 'Config 2: ReLU, Batch 64', 'activation': 'relu', 'batch_size': 64},
    {'name': 'Config 3: ReLU, Batch 128', 'activation': 'relu', 'batch_size': 128},
    {'name': 'Config 4: Tanh, Batch 32', 'activation': 'tanh', 'batch_size': 32},
    {'name': 'Config 5: Tanh, Batch 64', 'activation': 'tanh', 'batch_size': 64},
    {'name': 'Config 6: Sigmoid, Batch 32', 'activation': 'sigmoid', 'batch_size': 32},
]

print("Testing 6 configurations:")
for i, config in enumerate(configs, 1):
    print(f"{i}. {config['name']}")

In [ ]:
# Train models with different configurations
results = []

for config in configs:
    print(f"\n{'='*60}")
    print(f"Training: {config['name']}")
    print(f"{'='*60}")
    
    # Build model
    model = Sequential([
        Embedding(max_words, 64, input_length=max_len),
        GlobalAveragePooling1D(),
        Dense(128, activation=config['activation']),
        Dropout(0.3),
        Dense(64, activation=config['activation']),
        Dropout(0.3),
        Dense(1, activation='sigmoid')  # Binary output
    ])
    
    # Compile model
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    # Train model
    history = model.fit(
        X_train_pad, y_train,
        epochs=5,
        batch_size=config['batch_size'],
        validation_split=0.2,
        verbose=0
    )
    
    # Predict on test set
    y_pred_prob = model.predict(X_test_pad, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Store results
    results.append({
        'Config': config['name'],
        'Activation': config['activation'],
        'Batch Size': config['batch_size'],
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })
    
    print(f"✓ Accuracy: {accuracy:.4f}")
    print(f"✓ Precision: {precision:.4f}")
    print(f"✓ Recall: {recall:.4f}")
    print(f"✓ F1 Score: {f1:.4f}")

print("\n" + "="*60)
print("All configurations trained!")
print("="*60)

In [ ]:
# Display results table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Accuracy', ascending=False)
print("\nRESULTS TABLE (Sorted by Accuracy):")
print(results_df.to_string(index=False))

# Identify best model
best_config = results_df.iloc[0]
print(f"\n{'='*60}")
print("BEST MODEL:")
print(f"Configuration: {best_config['Config']}")
print(f"Accuracy: {best_config['Accuracy']:.4f}")
print(f"Precision: {best_config['Precision']:.4f}")
print(f"Recall: {best_config['Recall']:.4f}")
print(f"F1 Score: {best_config['F1 Score']:.4f}")
print(f"{'='*60}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

for idx, (metric, color) in enumerate(zip(metrics, colors)):
    ax = axes[idx // 2, idx % 2]
    ax.barh(results_df['Config'], results_df[metric], color=color, edgecolor='black')
    ax.set_xlabel(metric, fontweight='bold', fontsize=12)
    ax.set_title(f'{metric} Comparison', fontweight='bold', fontsize=14)
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    for i, v in enumerate(results_df[metric]):
        ax.text(v, i, f' {v:.4f}', va='center')

plt.tight_layout()
plt.savefig('assignment2_results.png', dpi=300, bbox_inches='tight')
plt.show()
print("Visualization saved as 'assignment2_results.png'")

In [ ]:
# Train best model for custom testing
best_activation = best_config['Activation']
best_batch_size = int(best_config['Batch Size'])

print(f"Training final model with best configuration...")
print(f"Activation: {best_activation}, Batch Size: {best_batch_size}")

final_model = Sequential([
    Embedding(max_words, 64, input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(128, activation=best_activation),
    Dropout(0.3),
    Dense(64, activation=best_activation),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

final_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
final_model.fit(X_train_pad, y_train, epochs=5, batch_size=best_batch_size, 
                validation_split=0.2, verbose=0)

print("✓ Final model trained!")

In [ ]:
# Test with custom reviews (not from dataset)
print("\n" + "="*60)
print("TESTING WITH CUSTOM REVIEWS")
print("="*60)

custom_reviews = [
    "This movie was absolutely fantastic! The acting was superb and the plot kept me engaged throughout. Highly recommend!",
    "Terrible waste of time. Poor acting, weak storyline, and boring scenes. Would not recommend to anyone.",
    "An amazing masterpiece! Brilliant direction and outstanding performances. One of the best films I've ever seen.",
    "Disappointing and dull. The movie failed to deliver on its promise. Very underwhelming experience."
]

expected_sentiments = ["Positive", "Negative", "Positive", "Negative"]

# Preprocess custom reviews
custom_seq = tokenizer.texts_to_sequences(custom_reviews)
custom_pad = pad_sequences(custom_seq, maxlen=max_len)

# Predict
predictions = final_model.predict(custom_pad, verbose=0)

print("\nCustom Review Predictions:")
print("="*60)
for i, (review, pred, expected) in enumerate(zip(custom_reviews, predictions, expected_sentiments), 1):
    sentiment = "Positive" if pred[0] > 0.5 else "Negative"
    confidence = pred[0] if pred[0] > 0.5 else 1 - pred[0]
    match = "✓" if sentiment == expected else "✗"
    
    print(f"\nReview {i}:")
    print(f"Text: {review[:80]}...")
    print(f"Predicted: {sentiment} (Confidence: {confidence[0]:.2%}) {match}")
    print(f"Expected: {expected}")
    print("-" * 60)

In [ ]:
# Test with YOUR OWN review
print("\n" + "="*60)
print("TEST YOUR OWN REVIEW!")
print("="*60)

# Change this to your own movie review
your_review = "The cinematography was breathtaking and the soundtrack was incredible. A must-watch film!"

# Preprocess
your_seq = tokenizer.texts_to_sequences([your_review])
your_pad = pad_sequences(your_seq, maxlen=max_len)

# Predict
your_pred = final_model.predict(your_pad, verbose=0)[0][0]
your_sentiment = "Positive 😊" if your_pred > 0.5 else "Negative 😞"
your_confidence = your_pred if your_pred > 0.5 else 1 - your_pred

print(f"\nYour Review: {your_review}")
print(f"\nPredicted Sentiment: {your_sentiment}")
print(f"Confidence: {your_confidence:.2%}")
print("="*60)

In [ ]:
# Save results
results_df.to_csv('assignment2_results.csv', index=False)
print("\nResults saved to 'assignment2_results.csv'")
print("\n✓ Assignment 2 Complete!")